  # Детектор переломов

В ноутбуке обучается бинарный детектор переломов. В ноутбуке было проведено 3 эксперимента с перебором моделей, обучение итоговой модели не отражено именно в этом ноутбуке

# Импорты

In [ ]:
import subprocess
import sys
from collections import Counter
from pathlib import Path
import json
import random
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from PIL import Image, ImageDraw
from ultralytics import YOLO

In [ ]:
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "ultralytics==8.4.92",
        "kagglehub",
    ]
)

In [ ]:
SEED = 42
TARGET_MAP50 = 0.50

MODEL_CANDIDATES = ("yolo26m-seg.pt", "yolo26l-seg.pt")
PILOT_EPOCHS = 12
FINETUNE_EPOCHS = 70
IMGSZ = 768
PATIENCE = 20
MIN_PILOT_MAP50 = 0.05

GPU_COUNT = torch.cuda.device_count()
TRAIN_DEVICE = [0, 1] if GPU_COUNT >= 2 else 0
VAL_DEVICE = 0

# Учим на двух T4 на Kaggle
BATCH = 8 if GPU_COUNT >= 2 else 0.80

OUTPUT_DIR = Path.cwd() / "fracture_detector"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("torch:", torch.__version__)
print("ultralytics candidates:", MODEL_CANDIDATES)
print("output:", OUTPUT_DIR.resolve())
print("Количество GPU:", GPU_COUNT)
print("GPU для обучения:", TRAIN_DEVICE)

for index in range(GPU_COUNT):
    print(f"GPU {index}: {torch.cuda.get_device_name(index)}")

  ## Загрузка и датасета

In [ ]:
# Download latest version. Если датасет уже скачан, вернётся путь к кешу.
path = kagglehub.dataset_download(
    "pkdarabi/bone-fracture-detection-computer-vision-project"
)
path = Path(path)

print("Path to dataset files:", path)


def is_yolo_dataset(folder: Path) -> bool:
    """Проверяет наличие трёх YOLO-сплитов с images и labels."""
    return all(
        (folder / split / "images").is_dir() and (folder / split / "labels").is_dir()
        for split in ("train", "valid", "test")
    )


# Проверки
dataset_candidates = [
    yaml_file.parent
    for yaml_file in path.rglob("data.yaml")
    if is_yolo_dataset(yaml_file.parent)
]
if is_yolo_dataset(path):
    dataset_candidates.append(path)

if not dataset_candidates:
    raise FileNotFoundError(f"В {path} не найден датасет со сплитами train/valid/test.")

DATASET_ROOT = max(
    set(dataset_candidates),
    key=lambda folder: len(list((folder / "train" / "images").glob("*"))),
)

print("Selected dataset:", DATASET_ROOT)

  ## Проверка разметки


In [ ]:
# В этом блоке была попытка устранить leakage тестовых снимков (в итоге ухудшило метрику)
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CLASS_NAMES = [
    "elbow positive",
    "fingers positive",
    "forearm fracture",
    "humerus fracture",
    "humerus",
    "shoulder fracture",
    "wrist positive",
]


def audit_split(split: str) -> dict:
    """Проверяет пары image/label и возвращает статистику сплита."""
    image_dir = DATASET_ROOT / split / "images"
    label_dir = DATASET_ROOT / split / "labels"

    image_stems = {
        file.stem
        for file in image_dir.iterdir()
        if file.suffix.lower() in IMAGE_SUFFIXES
    }
    label_files = list(label_dir.glob("*.txt"))
    label_stems = {file.stem for file in label_files}

    missing_labels = image_stems - label_stems
    missing_images = label_stems - image_stems
    if missing_labels or missing_images:
        raise ValueError(
            f"{split}: missing labels={len(missing_labels)}, "
            f"missing images={len(missing_images)}"
        )

    class_counts = Counter()
    empty_labels = 0
    objects = 0

    for label_file in label_files:
        rows = [line.split() for line in label_file.read_text().splitlines() if line]
        empty_labels += len(rows) == 0

        for row in rows:
            # Segmentation label: class_id + минимум три пары (x, y).
            if len(row) < 7 or len(row) % 2 == 0:
                raise ValueError(f"Некорректная строка в {label_file}: {row}")

            class_id = int(row[0])
            coordinates = np.asarray(row[1:], dtype=float)
            if class_id not in range(len(CLASS_NAMES)):
                raise ValueError(f"Неизвестный class_id={class_id} в {label_file}")
            if np.any((coordinates < 0) | (coordinates > 1)):
                raise ValueError(f"Координаты вне [0, 1] в {label_file}")

            class_counts[class_id] += 1
            objects += 1

    return {
        "split": split,
        "images": len(image_stems),
        "objects": objects,
        "negative_images": empty_labels,
        "class_counts": dict(sorted(class_counts.items())),
    }


dataset_stats = [audit_split(split) for split in ("train", "valid", "test")]
print("Исходные Roboflow-сплиты:")
for stats in dataset_stats:
    print(
        f"{stats['split']:>5}: images={stats['images']:4d}, "
        f"objects={stats['objects']:4d}, "
        f"negative={stats['negative_images']:4d}, "
        f"classes={stats['class_counts']}"
    )


def source_id(image_path: Path) -> str:
    """Возвращает имя исходного снимка без Roboflow-хеша и расширения."""
    return image_path.stem.split(".rf.", maxsplit=1)[0]


def split_images(split: str) -> list[Path]:
    image_dir = DATASET_ROOT / split / "images"
    return sorted(
        path.resolve()
        for path in image_dir.iterdir()
        if path.suffix.lower() in IMAGE_SUFFIXES
    )


raw_images = {split: split_images(split) for split in ("train", "valid", "test")}
raw_sources = {
    split: {source_id(image_path) for image_path in images}
    for split, images in raw_images.items()
}

print("\nПересечения исходников до очистки:")
print("train/valid:", len(raw_sources["train"] & raw_sources["valid"]))
print("train/test: ", len(raw_sources["train"] & raw_sources["test"]))
print("valid/test: ", len(raw_sources["valid"] & raw_sources["test"]))

clean_source_ids = {
    "test": raw_sources["test"],
    "valid": raw_sources["valid"] - raw_sources["test"],
    "train": raw_sources["train"] - raw_sources["valid"] - raw_sources["test"],
}
clean_images = {
    split: [
        image_path
        for image_path in raw_images[split]
        if source_id(image_path) in clean_source_ids[split]
    ]
    for split in ("train", "valid", "test")
}

assert clean_source_ids["train"].isdisjoint(clean_source_ids["valid"])
assert clean_source_ids["train"].isdisjoint(clean_source_ids["test"])
assert clean_source_ids["valid"].isdisjoint(clean_source_ids["test"])
assert all(clean_images.values()), "После очистки один из сплитов оказался пустым"


def label_path(image_path: Path) -> Path:
    """Находит YOLO-label для изображения из любой исходной папки сплита."""
    return image_path.parent.parent / "labels" / f"{image_path.stem}.txt"


def audit_manifest(split: str, images: list[Path]) -> dict:
    objects = 0
    negative_images = 0
    for image_path in images:
        labels = label_path(image_path)
        if not labels.is_file():
            raise FileNotFoundError(f"Не найден label для {image_path}")
        rows = [line for line in labels.read_text().splitlines() if line.strip()]
        objects += len(rows)
        negative_images += len(rows) == 0
    return {
        "split": split,
        "images": len(images),
        "sources": len(clean_source_ids[split]),
        "objects": objects,
        "negative_images": negative_images,
    }


MANIFEST_DIR = OUTPUT_DIR / "manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
manifest_paths = {}
clean_stats = []
for split in ("train", "valid", "test"):
    manifest_path = MANIFEST_DIR / f"{split}.txt"
    manifest_path.write_text(
        "\n".join(str(path) for path in clean_images[split]) + "\n",
        encoding="utf-8",
    )
    manifest_paths[split] = manifest_path.resolve()
    clean_stats.append(audit_manifest(split, clean_images[split]))

print("\nОчищенные group-exclusive сплиты:")
for stats in clean_stats:
    print(
        f"{stats['split']:>5}: images={stats['images']:4d}, "
        f"sources={stats['sources']:4d}, objects={stats['objects']:4d}, "
        f"negative={stats['negative_images']:4d}"
    )


DATA_YAML = OUTPUT_DIR / "fracture_data.yaml"
data_config = {
    "train": str(manifest_paths["train"]),
    "val": str(manifest_paths["valid"]),
    "test": str(manifest_paths["test"]),
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}
DATA_YAML.write_text(
    yaml.safe_dump(data_config, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

print("Generated YAML:", DATA_YAML)
print(DATA_YAML.read_text(encoding="utf-8"))

  ## Визуальная проверка нескольких объектов

In [ ]:
def show_annotations(split: str = "train", count: int = 6) -> None:
    """Рисует полигоны и их bounding boxes на случайных снимках"""
    positive_images = [
        path for path in clean_images[split] if label_path(path).stat().st_size
    ]
    selected = random.sample(positive_images, min(count, len(positive_images)))

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for axis, image_file in zip(axes.flat, selected):
        label_file = label_path(image_file)
        image = Image.open(image_file).convert("RGB")
        draw = ImageDraw.Draw(image)
        width, height = image.size

        for line in label_file.read_text().splitlines():
            row = line.split()
            coordinates = np.asarray(row[1:], dtype=float).reshape(-1, 2)
            points = [(int(x * width), int(y * height)) for x, y in coordinates]
            draw.line(points + [points[0]], fill="lime", width=max(2, width // 300))
            xs, ys = zip(*points)
            draw.rectangle(
                (min(xs), min(ys), max(xs), max(ys)),
                outline="red",
                width=max(2, width // 300),
            )

        axis.imshow(image)
        axis.set_title(image_file.name[:45])
        axis.axis("off")

    for axis in axes.flat[len(selected) :]:
        axis.axis("off")
    plt.tight_layout()
    plt.show()


show_annotations()

  ## Проверка GPU

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("GPU не обнаружен")

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name}, VRAM: {gpu_memory_gb:.1f} GB")

## Пилотные эксперименты

In [ ]:
def evaluate(weights: Path, split: str, imgsz: int):
    """Считает box и mask mAP без test-time augmentation"""
    evaluated_model = YOLO(str(weights))
    metrics = evaluated_model.val(
        data=str(DATA_YAML),
        split=split,
        single_cls=True,
        imgsz=imgsz,
        batch=8,
        device=VAL_DEVICE,
        workers=2,
        conf=0.001,
        iou=0.70,
        plots=True,
        project=str(OUTPUT_DIR / "runs"),
        name=f"{split}_{weights.parent.parent.name}_{imgsz}",
        exist_ok=False,
        verbose=True,
    )
    print(
        f"{split}: box mAP@0.5={metrics.box.map50:.4f}, "
        f"box mAP@0.5:0.95={metrics.box.map:.4f}, "
        f"mask mAP@0.5={metrics.seg.map50:.4f}"
    )
    return metrics

In [ ]:
pilot_results = []
for candidate in MODEL_CANDIDATES:
    candidate_stem = Path(candidate).stem
    print(f"\n{'=' * 20} PILOT: {candidate} {'=' * 20}")
    pilot_model = YOLO(candidate)
    pilot_model.train(
        data=str(DATA_YAML),
        single_cls=True,
        epochs=PILOT_EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=TRAIN_DEVICE,
        workers=2,
        patience=PILOT_EPOCHS,
        optimizer="AdamW",
        lr0=1e-3,
        lrf=0.10,
        weight_decay=5e-4,
        warmup_epochs=2.0,
        freeze=10,
        cos_lr=True,
        close_mosaic=0,
        amp=True,
        cache=False,
        mask_ratio=2,
        overlap_mask=True,
        seed=SEED,
        deterministic=True,
        degrees=2.0,
        translate=0.02,
        scale=0.10,
        shear=0.0,
        perspective=0.0,
        fliplr=0.5,
        flipud=0.0,
        hsv_h=0.0,
        hsv_s=0.0,
        hsv_v=0.10,
        mosaic=0.0,
        mixup=0.0,
        copy_paste=0.0,
        plots=True,
        save=True,
        save_period=4,
        project=str(OUTPUT_DIR / "runs"),
        name=f"{candidate_stem}_pilot",
        exist_ok=False,
        verbose=True,
    )

    pilot_weights = Path(pilot_model.trainer.best)
    pilot_metrics = evaluate(pilot_weights, split="val", imgsz=IMGSZ)
    pilot_results.append(
        {
            "model": candidate,
            "weights": pilot_weights,
            "metrics": pilot_metrics,
            "box_map50": float(pilot_metrics.box.map50),
            "mask_map50": float(pilot_metrics.seg.map50),
        }
    )

pilot_results.sort(key=lambda result: result["box_map50"], reverse=True)
print("\nРезультаты pilot-экспериментов:")
for result in pilot_results:
    print(
        f"{result['model']}: box mAP@0.5={result['box_map50']:.4f}, "
        f"mask mAP@0.5={result['mask_map50']:.4f}"
    )

pilot_winner = pilot_results[0]
if pilot_winner["box_map50"] < MIN_PILOT_MAP50:
    raise RuntimeError(
        f"Лучший pilot дал box mAP@0.5={pilot_winner['box_map50']:.4f}, "
        f"что ниже порога {MIN_PILOT_MAP50:.2f}. Полный fine-tuning не запущен: "
        "сначала нужно менять данные или постановку задачи."
    )

print(
    f"\nПобедитель pilot: {pilot_winner['model']} "
    f"с box mAP@0.5={pilot_winner['box_map50']:.4f}"
)

## Файнтюн лучшей модели

In [ ]:
finetune_model = YOLO(str(pilot_winner["weights"]))
finetune_model.train(
    data=str(DATA_YAML),
    single_cls=True,
    epochs=FINETUNE_EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=TRAIN_DEVICE,
    workers=2,
    patience=PATIENCE,
    optimizer="AdamW",
    lr0=3e-4,
    lrf=0.05,
    weight_decay=5e-4,
    warmup_epochs=1.0,
    freeze=None,
    cos_lr=True,
    close_mosaic=10,
    amp=True,
    cache=False,
    mask_ratio=2,
    overlap_mask=True,
    seed=SEED,
    deterministic=True,
    degrees=3.0,
    translate=0.03,
    scale=0.15,
    shear=0.0,
    perspective=0.0,
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.15,
    mosaic=0.15,
    mixup=0.0,
    copy_paste=0.0,
    plots=True,
    save=True,
    save_period=5,
    project=str(OUTPUT_DIR / "runs"),
    name=f"{Path(pilot_winner['model']).stem}_full",
    exist_ok=False,
    verbose=True,
)

finetune_best = Path(finetune_model.trainer.best)
finetune_val_metrics = evaluate(finetune_best, split="val", imgsz=IMGSZ)

best_weights = pilot_winner["weights"]
best_val_metrics = pilot_winner["metrics"]
if finetune_val_metrics.box.map50 > pilot_winner["box_map50"]:
    best_weights = finetune_best
    best_val_metrics = finetune_val_metrics
    print("Для финала выбран результат полного fine-tuning.")
else:
    print("Pilot оказался лучше; его веса сохранены для финала.")

final_imgsz = IMGSZ

## Оценка на тесте

In [ ]:
test_metrics = evaluate(best_weights, split="test", imgsz=final_imgsz)

final_weights = OUTPUT_DIR / "fracture_detector_best.pt"
final_model = YOLO(str(best_weights))
final_model.model.names = {0: "fracture"}
final_model.save(str(final_weights))

summary = {
    "model": pilot_winner["model"],
    "task": "instance_segmentation",
    "binary_detection": True,
    "image_size": final_imgsz,
    "pilot_results": {
        result["model"]: {
            "box_map50": result["box_map50"],
            "mask_map50": result["mask_map50"],
        }
        for result in pilot_results
    },
    "split_policy": "source-group-exclusive, priority test > valid > train",
    "clean_splits": {
        stats["split"]: {key: value for key, value in stats.items() if key != "split"}
        for stats in clean_stats
    },
    "validation_map50": float(best_val_metrics.box.map50),
    "validation_map50_95": float(best_val_metrics.box.map),
    "validation_mask_map50": float(best_val_metrics.seg.map50),
    "validation_mask_map50_95": float(best_val_metrics.seg.map),
    "test_map50": float(test_metrics.box.map50),
    "test_map50_95": float(test_metrics.box.map),
    "test_mask_map50": float(test_metrics.seg.map50),
    "test_mask_map50_95": float(test_metrics.seg.map),
    "target_map50": TARGET_MAP50,
    "target_reached_on_test": bool(test_metrics.box.map50 >= TARGET_MAP50),
    "weights": str(final_weights.resolve()),
}
(OUTPUT_DIR / "metrics.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2, ensure_ascii=False))
if summary["target_reached_on_test"]:
    print(f"ЦЕЛЬ ДОСТИГНУТА: test mAP@0.5 = {test_metrics.box.map50:.4f}")
else:
    print(f"Цель не достигнута: test mAP@0.5 = {test_metrics.box.map50:.4f}. ")

print("Final weights:", final_weights.resolve())

  ## Примеры предсказаний

In [ ]:
final_model = YOLO(str(final_weights))
test_images = clean_images["test"]
preview_images = random.sample(test_images, min(12, len(test_images)))

prediction_results = final_model.predict(
    source=[str(file) for file in preview_images],
    imgsz=final_imgsz,
    conf=0.25,
    iou=0.70,
    device=VAL_DEVICE,
    save=True,
    project=str(OUTPUT_DIR / "predictions"),
    name="test_preview",
    exist_ok=True,
    verbose=False,
)

fig, axes = plt.subplots(3, 4, figsize=(16, 13))
for axis, result in zip(axes.flat, prediction_results):
    # result.plot() возвращает BGR, поэтому меняем порядок каналов.
    axis.imshow(result.plot()[..., ::-1])
    axis.set_title(Path(result.path).name[:38])
    axis.axis("off")
for axis in axes.flat[len(prediction_results) :]:
    axis.axis("off")
plt.tight_layout()
plt.show()

  ## Инференс на новых снимках

In [ ]:
def predict_fractures(source, confidence: float = 0.25, save: bool = True):
    """Запускает сохранённый детектор на файле, папке или списке изображений."""
    return final_model.predict(
        source=source,
        imgsz=final_imgsz,
        conf=confidence,
        iou=0.70,
        device=VAL_DEVICE,
        save=save,
        project=str(OUTPUT_DIR / "predictions"),
        name="inference",
        exist_ok=True,
    )